# Exploratory Data Analysis of Primock57 Audio

This notebook explores a sample consultation recording from the Primock57 dataset before diarization. The analysis focuses on audio loading, playback, waveform inspection, silence trimming, and time-frequency representations that help understand the structure of the speech signal.

## Import Libraries

Load the audio-processing, numerical, and visualization libraries used throughout the EDA. `librosa` is used for signal processing, while Matplotlib and Seaborn are used for plotting.

In [ ]:
import os
from os.path import isdir, join
from pathlib import Path
import pandas as pd

# Math
import numpy as np
from scipy.fftpack import fft
from scipy import signal
from scipy.io import wavfile
import librosa

from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import IPython.display as ipd
import librosa.display

import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
import pandas as pd

%matplotlib inline

## Install Audio Backend

`ffmpeg` provides audio decoding support in the Colab environment. Run this cell when working in a fresh notebook runtime.

In [ ]:
!apt-get install ffmpeg

## Configure Plotting Utilities

Set up the plotting style, color cycle, and repeated imports used by the visualization cells below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns

from glob import glob

import librosa
import librosa.display
import IPython.display as ipd

from itertools import cycle

sns.set_theme(style="white", palette=None)
color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

## Select a Sample Audio File

Point the notebook to one Primock57 consultation audio file. Update this path if the dataset is mounted somewhere else in Drive or on a local machine.

In [ ]:
audio_files = glob('/content/drive/MyDrive/Speech_assignment/audio/day1_consultation01_patient.wav')

## Listen to the Recording

Play the selected audio sample to get a qualitative sense of the recording, speaker turns, pauses, and background noise before visual analysis.

In [ ]:
ipd.Audio(audio_files[0])

## Load the Signal

Load the waveform samples and sample rate with `librosa`. The printed values help confirm that the audio was read correctly and show the shape of the signal array.

In [ ]:
samples, sample_rate = librosa.load(audio_files[0])
print(f'samples: {samples[:50]}')
print(f'anotehr samples: {samples[30000:30050]}')
print(f'shape of samples: {samples.shape}')
print(f'sample_rate: {sample_rate}')

## Waveform Analysis

Plot the raw waveform to inspect amplitude changes over time. This gives a first view of speech regions, silence, and possible signal variation in the consultation audio.

In [ ]:

pd.Series(samples).plot(figsize=(10, 5),
                        lw=1,
                        title='Raw Audio Example',
                        color=color_pal[0])
plt.show()

## Silence Trimming

Remove leading and trailing low-energy regions. This helps isolate the active portion of the recording and makes later analysis less affected by long silences.

In [ ]:
# Trimming leading/lagging silence
samples_trimmed, _ = librosa.effects.trim(samples, top_db=20)
pd.Series(samples_trimmed).plot(figsize=(10, 5),
                                lw=1,
                                title='Raw Audio Trimmed Example',
                                color=color_pal[1])
plt.show()

## Zoomed Waveform View

Inspect a short slice of the signal at sample level. A zoomed view makes local amplitude variation easier to see than the full recording plot.

In [ ]:
pd.Series(samples[30000:30500]).plot(figsize=(10, 5),
                                     lw=1,
                                     title='Raw Audio Zoomed In Example',
                                     color=color_pal[2])
plt.show()

## Short-Time Fourier Transform

Convert the waveform into a time-frequency representation with STFT, then map amplitudes to decibels for easier visualization.

In [ ]:
D = librosa.stft(samples)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
S_db.shape

## Spectrogram

Plot the STFT result as a spectrogram. This view shows how frequency energy changes over time and is useful for spotting speech structure and silence regions.

In [ ]:
# Plot the transformed audio data
fig, ax = plt.subplots(figsize=(10, 5))
img = librosa.display.specshow(S_db,
                              x_axis='time',
                              y_axis='log',
                              ax=ax)

ax.set_title('Spectrogram Example', fontsize=20)
fig.colorbar(img, ax=ax, format=f'%0.2f')
plt.show()

## Mel-Spectrogram Features

Compute a mel-scaled spectrogram, which compresses frequency information into a scale closer to human hearing and is commonly used in speech-processing models.

In [ ]:
S = librosa.feature.melspectrogram(y=samples,
                                   sr=sample_rate)
S_db_mel = librosa.amplitude_to_db(S, ref=np.max)

## Mel-Spectrogram Plot

Visualize the mel-spectrogram to review the speech signal in a model-friendly time-frequency representation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
# Plot the mel spectogram
img = librosa.display.specshow(S_db_mel,
                              x_axis='time',
                              y_axis='log',
                              ax=ax)
ax.set_title('Mel Spectogram Example', fontsize=20)
fig.colorbar(img, ax=ax, format=f'%0.2f')
plt.show()